In [1]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier
from sklift.models import SoloModel

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything # Import hàm seed_everything của bạn

# Tắt bớt log của Optuna để console không bị rối khi chạy nhiều seed
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu (Chỉ tải 1 lần ở ngoài vòng lặp)
print("Loading data...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion' # BÀI TOÁN CONVERSION
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test
X_train = train_df[in_features]
y_train = train_df[label_feature]
t_train = train_df[treatment_feature]

X_val = val_df[in_features]
y_val_true = val_df[label_feature].values.flatten()
t_val_true = val_df[treatment_feature].values.flatten()

X_test = test_df[in_features]
y_test_true = test_df[label_feature].values.flatten()
t_test_true = test_df[treatment_feature].values.flatten()

# 2. Khởi tạo danh sách các seed và list để lưu kết quả
seeds = [412312, 42, 1874, 902745, 1]
results = []

# 3. Chạy vòng lặp qua từng Seed
for SEED in seeds:
    print(f"\n{'='*50}")
    print(f"🚀 BẮT ĐẦU CHẠY VỚI SEED: {SEED}")
    print(f"{'='*50}")
    
    # Cố định seed cho môi trường
    seed_everything(SEED)

    # Hàm Objective định nghĩa bên trong để bắt được biến SEED hiện tại
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'num_leaves': trial.suggest_int('num_leaves', 10, 150),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'random_state': SEED, # Sử dụng current SEED
            'verbose': -1
        }
        
        # Khởi tạo Base Model là LGBMClassifier
        base_model = LGBMClassifier(**params)
        s_learner = SoloModel(estimator=base_model)
        
        s_learner.fit(X=X_train, y=y_train, treatment=t_train)
        uplift_pred_val = s_learner.predict(X_val)
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        return score

    # Chạy Optuna
    print("🔃 Đang chạy Optuna Tuning...")
    fixed_sampler = TPESampler(seed=SEED)
    study = optuna.create_study(direction="maximize", study_name=f"S_Learner_LGBM_Conv_{SEED}", sampler=fixed_sampler)
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    
    val_best_auqc = study.best_value
    best_params = study.best_params
    print(f"✅ Tuning hoàn tất! Best Val AUQC: {val_best_auqc:.4f}")

    # Huấn luyện mô hình Final
    print("🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...")
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['verbose'] = -1

    final_base_model = LGBMClassifier(**best_params_final)
    final_s_learner = SoloModel(estimator=final_base_model)
    final_s_learner.fit(X=X_train, y=y_train, treatment=t_train)

    # Dự đoán trên Test
    uplift_pred_test = final_s_learner.predict(X_test)

    # Đánh giá (Tắt plot=False để không in quá nhiều biểu đồ)
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    print(f"   Test AUUC: {auuc_score:.3f}")
    print(f"   Test AUQC: {auqc_score:.3f}")
    print(f"   Test Lift: {lift_score:.3f}")
    print(f"   Test KRCC: {krcc_score:.3f}")

    # Lưu kết quả của seed này vào dictionary
    run_result = {
        'seed': SEED,
        'val_best_auqc': val_best_auqc,
        'test_auuc': auuc_score,
        'test_auqc': auqc_score,
        'test_lift': lift_score,
        'test_krcc': krcc_score
    }
    # Gộp thêm các best parameters vào dict
    run_result.update(best_params)
    results.append(run_result)

# 4. Xuất kết quả ra file CSV
print("\n" + "="*50)
print("💾 ĐANG LƯU KẾT QUẢ VÀO FILE CSV...")
results_df = pd.DataFrame(results)
csv_filename = "s_learner_conversion_results.csv" # Đổi tên file cho Conversion
results_df.to_csv(csv_filename, index=False)
print(f"✅ Đã lưu thành công tại: {csv_filename}")

# Hiển thị thử bảng kết quả
display(results_df)

/home/datnghiemxuan/hai/.venv_3_12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...

🚀 BẮT ĐẦU CHẠY VỚI SEED: 412312
Locked random seed: 412312
🔃 Đang chạy Optuna Tuning...


Best trial: 12. Best value: 0.738426: 100%|██████████| 50/50 [00:39<00:00,  1.27it/s]


✅ Tuning hoàn tất! Best Val AUQC: 0.7384
🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...
   Test AUUC: 0.539
   Test AUQC: 0.537
   Test Lift: 0.009
   Test KRCC: 0.079

🚀 BẮT ĐẦU CHẠY VỚI SEED: 42
Locked random seed: 42
🔃 Đang chạy Optuna Tuning...


Best trial: 34. Best value: 0.837679: 100%|██████████| 50/50 [00:22<00:00,  2.21it/s]


...

Best trial: 25. Best value: 0.706115: 100%|██████████| 50/50 [00:26<00:00,  1.87it/s]


✅ Tuning hoàn tất! Best Val AUQC: 0.7061
🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...
   Test AUUC: 0.493
   Test AUQC: 0.492
   Test Lift: 0.007
   Test KRCC: -0.023

🚀 BẮT ĐẦU CHẠY VỚI SEED: 1
Locked random seed: 1
🔃 Đang chạy Optuna Tuning...


Best trial: 26. Best value: 0.723561: 100%|██████████| 50/50 [00:29<00:00,  1.69it/s]


✅ Tuning hoàn tất! Best Val AUQC: 0.7236
🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...
   Test AUUC: 0.525
   Test AUQC: 0.524
   Test Lift: 0.007
   Test KRCC: 0.043

💾 ĐANG LƯU KẾT QUẢ VÀO FILE CSV...
✅ Đã lưu thành công tại: s_learner_conversion_results.csv


,seed,val_best_auqc,test_auuc,test_auqc,test_lift,test_krcc,n_estimators,learning_rate,max_depth,num_leaves,min_child_samples,subsample,colsample_bytree
0,412312,0.738426,0.539298,0.537439,0.009208,0.078527,175,0.002628,5,11,26,0.827605,0.680728
1,42,0.837679,0.531706,0.530346,0.008528,0.055066,138,0.001039,8,10,76,0.716746,0.611227
2,1874,0.754950,0.559055,0.558495,0.007987,0.099130,390,0.001621,6,10,127,0.953509,0.556046
3,902745,0.706115,0.493475,0.492282,0.007379,-0.022935,290,0.033261,3,114,183,0.686424,0.764013
4,1,0.723561,0.524684,0.523787,0.007299,0.042792,87,0.002660,3,42,187,0.606948,0.930483
